In [1]:
# ============================================
# CELL 1: Imports (NO SHAP - causes freeze!)
# ============================================
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries imported!")

✅ Libraries imported!


In [2]:
# ============================================
# CELL 2: Load Datasets
# ============================================
df_intrusion = pd.read_csv(r'C:\Users\Kshaunish\cyber-xai-ml-project\data\raw\intrusion_data.csv')
df_phishing = pd.read_csv(r'C:\Users\Kshaunish\cyber-xai-ml-project\data\raw\phishing_data.csv')

print(f"✅ Intrusion loaded: {df_intrusion.shape}")
print(f"✅ Phishing loaded: {df_phishing.shape}")

✅ Intrusion loaded: (692703, 79)
✅ Phishing loaded: (235795, 56)


In [3]:
# ============================================
# CELL 3: Clean Intrusion Dataset
# ============================================

# Step 1: Drop missing values
before = len(df_intrusion)
df_intrusion.dropna(inplace=True)
print(f"✅ Dropped {before - len(df_intrusion)} rows with missing values")

# Step 2: Drop duplicates
before = len(df_intrusion)
df_intrusion.drop_duplicates(inplace=True)
print(f"✅ Dropped {before - len(df_intrusion)} duplicate rows")

# Step 3: Replace infinite values
inf_count = np.isinf(df_intrusion.select_dtypes(include=[np.number])).sum().sum()
df_intrusion.replace([np.inf, -np.inf], np.nan, inplace=True)
df_intrusion.dropna(inplace=True)
print(f"✅ Removed {inf_count} infinite values")

print(f"\nFinal intrusion shape: {df_intrusion.shape}")

✅ Dropped 1008 rows with missing values
✅ Dropped 80962 duplicate rows
✅ Removed 482 infinite values

Final intrusion shape: (610492, 79)


In [4]:
# ============================================
# CELL 4: Clean Phishing Dataset
# ============================================

# Drop duplicates only (no missing values!)
before = len(df_phishing)
df_phishing.drop_duplicates(inplace=True)
print(f"✅ Dropped {before - len(df_phishing)} duplicate rows")

print(f"\nFinal phishing shape: {df_phishing.shape}")

✅ Dropped 0 duplicate rows

Final phishing shape: (235795, 56)


In [5]:
# ============================================
# CELL 5: Encode Labels
# ============================================

# Intrusion - convert attack names to numbers
le = LabelEncoder()
df_intrusion[' Label'] = le.fit_transform(df_intrusion[' Label'])

print("✅ Intrusion labels encoded!")
print("\nLabel mapping (name → number):")
for i, name in enumerate(le.classes_):
    print(f"  {name} → {i}")

# Phishing already numeric (0 and 1) - no encoding needed!
print("\n✅ Phishing labels already numeric - no encoding needed!")

✅ Intrusion labels encoded!

Label mapping (name → number):
  BENIGN → 0
  DoS GoldenEye → 1
  DoS Hulk → 2
  DoS Slowhttptest → 3
  DoS slowloris → 4
  Heartbleed → 5

✅ Phishing labels already numeric - no encoding needed!


In [6]:
# ============================================
# CELL 6: Separate Features & Labels
# ============================================

# Intrusion - separate features (X) from label (y)
X_intrusion = df_intrusion.drop(columns=[' Label'])
y_intrusion = df_intrusion[' Label']

# Phishing - separate features (X) from label (y)
X_phishing = df_phishing.drop(columns=['label'])
y_phishing = df_phishing['label']

print(f"✅ Intrusion — Features: {X_intrusion.shape}, Labels: {y_intrusion.shape}")
print(f"✅ Phishing  — Features: {X_phishing.shape}, Labels: {y_phishing.shape}")

✅ Intrusion — Features: (610492, 78), Labels: (610492,)
✅ Phishing  — Features: (235795, 55), Labels: (235795,)


In [8]:
# ============================================
# CELL 7: Feature Scaling
# ============================================

# First remove non-numeric columns from phishing
non_numeric = X_phishing.select_dtypes(exclude=[np.number]).columns.tolist()
print(f"Non-numeric columns in phishing (will be dropped): {non_numeric}")
X_phishing = X_phishing.drop(columns=non_numeric)
print(f"Phishing features after dropping non-numeric: {X_phishing.shape}")

# Now scale both datasets
scaler_intrusion = StandardScaler()
scaler_phishing = StandardScaler()

X_intrusion_scaled = scaler_intrusion.fit_transform(X_intrusion)
X_phishing_scaled = scaler_phishing.fit_transform(X_phishing)

print("\n✅ Intrusion features scaled!")
print("✅ Phishing features scaled!")
print(f"\nIntrusion scaled shape: {X_intrusion_scaled.shape}")
print(f"Phishing scaled shape: {X_phishing_scaled.shape}")

Non-numeric columns in phishing (will be dropped): ['FILENAME', 'URL', 'Domain', 'TLD', 'Title']
Phishing features after dropping non-numeric: (235795, 50)

✅ Intrusion features scaled!
✅ Phishing features scaled!

Intrusion scaled shape: (610492, 78)
Phishing scaled shape: (235795, 50)


In [9]:
# ============================================
# CELL 8: Train/Test Split
# ============================================

# Intrusion split
X_train_int, X_test_int, y_train_int, y_test_int = train_test_split(
    X_intrusion_scaled, y_intrusion, 
    test_size=0.2, 
    random_state=42,
    stratify=y_intrusion
)

# Phishing split
X_train_phi, X_test_phi, y_train_phi, y_test_phi = train_test_split(
    X_phishing_scaled, y_phishing,
    test_size=0.2,
    random_state=42,
    stratify=y_phishing
)

print("✅ Train/Test Split Done!")
print(f"\nIntrusion — Train: {X_train_int.shape}, Test: {X_test_int.shape}")
print(f"Phishing  — Train: {X_train_phi.shape}, Test: {X_test_phi.shape}")

✅ Train/Test Split Done!

Intrusion — Train: (488393, 78), Test: (122099, 78)
Phishing  — Train: (188636, 50), Test: (47159, 50)


In [10]:
# ============================================
# CELL 9: SMOTE - Fix Class Imbalance
# ============================================
print("Applying SMOTE to intrusion dataset...")
print("⚠️ This will take 5-10 minutes — please wait!")

smote = SMOTE(random_state=42)
X_train_int_smote, y_train_int_smote = smote.fit_resample(X_train_int, y_train_int)

print(f"\n✅ SMOTE applied!")
print(f"Before SMOTE: {X_train_int.shape[0]:,} samples")
print(f"After SMOTE:  {X_train_int_smote.shape[0]:,} samples")
print(f"\nClass distribution after SMOTE:")
import collections
counter = collections.Counter(y_train_int_smote)
for k, v in sorted(counter.items()):
    print(f"  Class {k}: {v:,}")

Applying SMOTE to intrusion dataset...
⚠️ This will take 5-10 minutes — please wait!

✅ SMOTE applied!
Before SMOTE: 488,393 samples
After SMOTE:  2,000,328 samples

Class distribution after SMOTE:
  Class 0: 333,388
  Class 1: 333,388
  Class 2: 333,388
  Class 3: 333,388
  Class 4: 333,388
  Class 5: 333,388


In [11]:
# ============================================
# CELL 10: Save Processed Data
# ============================================
import numpy as np

# Save intrusion data
np.save(r'C:\Users\Kshaunish\cyber-xai-ml-project\data\processed\X_train_int_smote.npy', X_train_int_smote)
np.save(r'C:\Users\Kshaunish\cyber-xai-ml-project\data\processed\y_train_int_smote.npy', y_train_int_smote)
np.save(r'C:\Users\Kshaunish\cyber-xai-ml-project\data\processed\X_test_int.npy', X_test_int)
np.save(r'C:\Users\Kshaunish\cyber-xai-ml-project\data\processed\y_test_int.npy', y_test_int)

# Save phishing data
np.save(r'C:\Users\Kshaunish\cyber-xai-ml-project\data\processed\X_train_phi.npy', X_train_phi)
np.save(r'C:\Users\Kshaunish\cyber-xai-ml-project\data\processed\y_train_phi.npy', y_train_phi)
np.save(r'C:\Users\Kshaunish\cyber-xai-ml-project\data\processed\X_test_phi.npy', X_test_phi)
np.save(r'C:\Users\Kshaunish\cyber-xai-ml-project\data\processed\y_test_phi.npy', y_test_phi)

print("✅ All processed data saved!")
print("\nSaved files:")
print("  Intrusion: X_train, y_train (SMOTE), X_test, y_test")
print("  Phishing:  X_train, y_train, X_test, y_test")

✅ All processed data saved!

Saved files:
  Intrusion: X_train, y_train (SMOTE), X_test, y_test
  Phishing:  X_train, y_train, X_test, y_test
